# PushT imitation learning benchmark: ACT (trained here) vs Diffusion Policy (pretrained)

This notebook:
1. Installs LeRobot and logs in to Weights & Biases.
2. Trains an **ACT** policy on the `lerobot/pusht` dataset (25k steps), with W&B logging and checkpoints saved to Google Drive.
3. Evaluates the trained ACT and the pretrained `lerobot/diffusion_pusht` on the **same 50 environment seeds**.
4. Produces a results table (success rate with 95% CI, paired significance test), a per-seed CSV and a list of failure cases with videos.

**Before you run anything**
- `Runtime > Change runtime type > T4 GPU` (or better), then Save.
- Add a Colab secret named `WANDB_API_KEY` (key icon in the left sidebar) with your key from https://wandb.ai/authorize, with notebook access enabled.
- Run the cells top to bottom. Colab disconnects idle tabs, so keep it open during training.

Runtime on a free T4: setup ~5 min, training ~45 min (~10 steps/s), evaluation ~15 min.

## 1. Check the GPU and Python version

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print("Python", sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.6.1 needs Python 3.12+"

## 2. Install LeRobot

Pinned to `lerobot==0.6.1` so the commands below keep working. This takes a few minutes.
If Colab asks to restart the session afterwards, restart and continue with step 3.

In [ ]:
!pip install -q "lerobot[training,pusht,diffusion,evaluation]==0.6.1"
!lerobot-train --help > /dev/null && echo "lerobot-train OK"

## 3. Settings, Google Drive and W&B login

Checkpoints and results are mirrored to Google Drive so they survive a Colab disconnect. Use a new `RUN_NAME` for each fresh training run.

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")

RUN_NAME = "act_pusht_25k"          # change for a new training run
STEPS = 25_000
SAVE_FREQ = 5_000
EVAL_START_SEED = 100_000           # episodes use seeds 100000..100049, same for every policy
N_EVAL_EPISODES = 50
WANDB_PROJECT = "lerobot-pusht-benchmark"

WORK = Path("/content/outputs")                                   # fast local disk
TRAIN_DIR = WORK / "train" / RUN_NAME
EVAL_DIR = WORK / "eval"
DRIVE_DIR = Path("/content/drive/MyDrive/lerobot-pusht-benchmark")
(DRIVE_DIR / "checkpoints" / RUN_NAME).mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["SDL_VIDEODRIVER"] = "dummy"   # headless rendering for the PushT sim
!wandb login --relogin "$WANDB_API_KEY" > /dev/null && echo "W&B login OK"

## 4. Back up checkpoints to Drive in the background

LeRobot writes checkpoints to local disk (Drive can't hold the `last` symlink it creates). This thread copies each finished checkpoint to Drive every 2 minutes and keeps the newest two there (each is about 600 MB because it includes the optimizer state).

In [ ]:
import shutil, threading, time

def backup_checkpoints(stop_event, keep=2, every_s=120):
    src_root = TRAIN_DIR / "checkpoints"
    dst_root = DRIVE_DIR / "checkpoints" / RUN_NAME
    while not stop_event.is_set():
        if src_root.exists():
            steps = sorted(p for p in src_root.iterdir() if p.is_dir() and not p.is_symlink() and p.name.isdigit())
            for ckpt in steps[:-1] if not stop_event.is_set() else steps:   # newest may still be writing
                dst = dst_root / ckpt.name
                if not dst.exists():
                    shutil.copytree(ckpt, dst.with_suffix(".tmp"), dirs_exist_ok=True)
                    dst.with_suffix(".tmp").rename(dst)
                    print(f"[backup] {ckpt.name} -> Drive")
            on_drive = sorted(p for p in dst_root.iterdir() if p.name.isdigit())
            for old in on_drive[:-keep]:
                shutil.rmtree(old)
        stop_event.wait(every_s)

stop_backup = threading.Event()
threading.Thread(target=backup_checkpoints, args=(stop_backup,), daemon=True).start()
print("Backup thread running")

## 5. Train ACT on PushT

What the flags mean:
- `--policy.type=act`: Action Chunking Transformer with default settings (ResNet-18 backbone, chunk of 100 actions, CVAE objective).
- `--env.type=pusht --env_eval_freq`: every 5k steps, roll out 10 episodes in the simulator and log success rate to W&B, so you get a learning curve in addition to the loss.
- `--wandb.disable_artifact=true`: checkpoints go to Drive instead of being uploaded to W&B.

The W&B run link is printed near the start of the log.

In [ ]:
!lerobot-train \
  --policy.type=act \
  --dataset.repo_id=lerobot/pusht \
  --env.type=pusht \
  --output_dir={TRAIN_DIR} \
  --job_name={RUN_NAME} \
  --steps={STEPS} \
  --save_freq={SAVE_FREQ} \
  --env_eval_freq={SAVE_FREQ} \
  --log_freq=200 \
  --eval.n_episodes=10 --eval.batch_size=10 --eval.use_async_envs=false \
  --batch_size=8 --num_workers=2 \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --wandb.enable=true --wandb.project={WANDB_PROJECT} --wandb.disable_artifact=true

### 5b. Only if the session died during training: resume

Run steps 1, 3 and 4 first, then uncomment and run this cell. It restores the newest checkpoint from Drive and continues training.

In [ ]:
# ckpts = sorted(p for p in (DRIVE_DIR / "checkpoints" / RUN_NAME).iterdir() if p.name.isdigit())
# latest = ckpts[-1]
# local = TRAIN_DIR / "checkpoints" / latest.name
# shutil.copytree(latest, local, dirs_exist_ok=True)
# (TRAIN_DIR / "checkpoints" / "last").unlink(missing_ok=True)
# (TRAIN_DIR / "checkpoints" / "last").symlink_to(latest.name)
# print("Resuming from", local)
# !lerobot-train --config_path={local}/pretrained_model/train_config.json --resume=true

When training has finished, stop the background backup (this copies the final checkpoint to Drive too).

In [ ]:
stop_backup.set()
time.sleep(5)
# final copy of every remaining checkpoint
for ckpt in sorted(p for p in (TRAIN_DIR / "checkpoints").iterdir() if p.is_dir() and not p.is_symlink()):
    dst = DRIVE_DIR / "checkpoints" / RUN_NAME / ckpt.name
    if not dst.exists():
        shutil.copytree(ckpt, dst)
        print("copied", ckpt.name)
FINAL_CKPT = TRAIN_DIR / "checkpoints" / f"{STEPS:06d}" / "pretrained_model"
print("Final checkpoint:", FINAL_CKPT, FINAL_CKPT.exists())

## 6. Evaluate both policies on the same 50 seeds

`--seed=100000` makes episode *i* start from seed 100000 + *i*, so both policies face exactly the same 50 initial T-block positions. A PushT episode counts as a success when the T covers at least 95% of the target. The first 10 episodes of each run are saved as videos.

In [ ]:
# Our ACT checkpoint
!lerobot-eval \
  --policy.path={FINAL_CKPT} \
  --env.type=pusht \
  --eval.n_episodes={N_EVAL_EPISODES} --eval.batch_size=10 --eval.use_async_envs=false \
  --seed={EVAL_START_SEED} \
  --policy.device=cuda --policy.use_amp=false \
  --output_dir={EVAL_DIR}/act

In [ ]:
# Pretrained Diffusion Policy from the LeRobot team (slower: it runs many denoising steps per action chunk)
!lerobot-eval \
  --policy.path=lerobot/diffusion_pusht \
  --env.type=pusht \
  --eval.n_episodes={N_EVAL_EPISODES} --eval.batch_size=10 --eval.use_async_envs=false \
  --seed={EVAL_START_SEED} \
  --policy.device=cuda --policy.use_amp=false \
  --output_dir={EVAL_DIR}/diffusion

### 6b. Optional ablation (5 minutes, no retraining): ACT with temporal ensembling

ACT executes all 100 predicted actions open-loop by default. PushT needs reactive corrections, so re-planning every step and averaging overlapping chunks (temporal ensembling, from the ACT paper) may change the result. This is an inference-time change only, evaluated on the same seeds.

In [ ]:
!lerobot-eval \
  --policy.path={FINAL_CKPT} \
  --policy.n_action_steps=1 --policy.temporal_ensemble_coeff=0.01 \
  --env.type=pusht \
  --eval.n_episodes={N_EVAL_EPISODES} --eval.batch_size=10 --eval.use_async_envs=false \
  --seed={EVAL_START_SEED} \
  --policy.device=cuda --policy.use_amp=false \
  --output_dir={EVAL_DIR}/act_ensemble

## 7. Compare the results

Writes `results.md` (table with success rate, Wilson 95% CI, average max reward, and an exact McNemar test on the paired seeds), `per_seed.csv` and `failures.md`, and logs the tables plus videos to W&B.

In [ ]:
%%writefile /content/compare_results.py
"""Compare LeRobot eval runs that were evaluated on the same seeds.

Reads the eval_info.json written by `lerobot-eval` for each run and produces:
  results/results.md   - summary table (success rate with 95% CI, avg max reward) + paired test
  results/per_seed.csv - one row per seed, success/max_reward for every run
  results/failures.md  - failed seeds per run, with the rendered video path when one exists

Usage:
  python compare_results.py \
      --run "ACT (ours, 25k steps)=outputs/eval/act" \
      --run "Diffusion (lerobot/diffusion_pusht)=outputs/eval/diffusion" \
      --start-seed 100000 --out results [--wandb-project lerobot-pusht-benchmark]
"""

import argparse
import csv
import json
import math
from pathlib import Path


def wilson_ci(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return float("nan"), float("nan")
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def mcnemar_exact_p(b: int, c: int) -> float:
    """Two-sided exact McNemar test on the discordant pairs (b, c)."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2**n
    return min(1.0, 2 * tail)


def load_run(eval_dir: Path, start_seed: int) -> dict:
    info = json.loads((eval_dir / "eval_info.json").read_text())
    tasks = info["per_task"]
    if len(tasks) != 1:
        raise ValueError(f"{eval_dir}: expected a single task (PushT), found {len(tasks)}")
    metrics = tasks[0]["metrics"]
    successes = [bool(s) for s in metrics["successes"]]
    max_rewards = [float(r) for r in metrics["max_rewards"]]
    # lerobot-eval seeds episode i with start_seed + i, and renders the first episodes in order.
    videos = dict(enumerate(metrics.get("video_paths", [])))
    return {
        "seeds": [start_seed + i for i in range(len(successes))],
        "successes": successes,
        "max_rewards": max_rewards,
        "videos": videos,
        "eval_s": info.get("overall", {}).get("eval_s"),
    }


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--run", action="append", required=True, help='"Label=path/to/eval_dir"')
    parser.add_argument("--start-seed", type=int, required=True, help="The --seed passed to lerobot-eval")
    parser.add_argument("--out", type=Path, default=Path("results"))
    parser.add_argument("--wandb-project", default=None, help="Also log the tables (and failure videos) to W&B")
    args = parser.parse_args()

    runs = {}
    for spec in args.run:
        label, _, path = spec.partition("=")
        runs[label] = load_run(Path(path), args.start_seed)

    seed_lists = {tuple(r["seeds"]) for r in runs.values()}
    if len(seed_lists) != 1:
        raise ValueError("Runs were not evaluated on the same number of episodes / seeds")
    seeds = runs[next(iter(runs))]["seeds"]
    n = len(seeds)
    args.out.mkdir(parents=True, exist_ok=True)

    # Summary table
    lines = [
        f"Evaluated on PushT, {n} episodes, seeds {seeds[0]}-{seeds[-1]} (identical for every policy).",
        "",
        "| Policy | Success rate | 95% CI (Wilson) | Avg max reward | Eval time (s) |",
        "|---|---|---|---|---|",
    ]
    for label, r in runs.items():
        k = sum(r["successes"])
        lo, hi = wilson_ci(k, n)
        avg_max = sum(r["max_rewards"]) / n
        eval_s = f"{r['eval_s']:.0f}" if r["eval_s"] is not None else "-"
        lines.append(f"| {label} | {k}/{n} = {100 * k / n:.0f}% | {100 * lo:.0f}-{100 * hi:.0f}% | {avg_max:.3f} | {eval_s} |")

    # Paired comparison between every pair of runs (same seeds -> McNemar)
    labels = list(runs)
    if len(labels) > 1:
        lines += ["", "Paired comparison on identical seeds (exact McNemar test):", ""]
        lines += ["| A | B | A ok / B fail | A fail / B ok | p-value |", "|---|---|---|---|---|"]
        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                a, b = runs[labels[i]]["successes"], runs[labels[j]]["successes"]
                only_a = sum(x and not y for x, y in zip(a, b))
                only_b = sum(y and not x for x, y in zip(a, b))
                lines.append(f"| {labels[i]} | {labels[j]} | {only_a} | {only_b} | {mcnemar_exact_p(only_a, only_b):.3f} |")
    (args.out / "results.md").write_text("\n".join(lines) + "\n")

    # Per-seed CSV
    with open(args.out / "per_seed.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["seed"] + [f"{l} {m}" for l in labels for m in ("success", "max_reward")])
        for i, seed in enumerate(seeds):
            writer.writerow([seed] + [v for l in labels for v in (int(runs[l]["successes"][i]), f"{runs[l]['max_rewards'][i]:.4f}")])

    # Failures
    fail_lines = ["# Failed episodes", ""]
    for label, r in runs.items():
        failed = [i for i, s in enumerate(r["successes"]) if not s]
        fail_lines.append(f"## {label}: {len(failed)} failures")
        fail_lines.append("")
        for i in failed:
            video = r["videos"].get(i)
            note = f" - video: `{video}`" if video else " - no video (only the first episodes are rendered)"
            fail_lines.append(f"- seed {seeds[i]} (episode {i}), max reward {r['max_rewards'][i]:.3f}{note}")
        fail_lines.append("")
    (args.out / "failures.md").write_text("\n".join(fail_lines))

    print((args.out / "results.md").read_text())
    print(f"Wrote {args.out}/results.md, per_seed.csv, failures.md")

    if args.wandb_project:
        import wandb

        run = wandb.init(project=args.wandb_project, name="pusht-benchmark-comparison", job_type="eval")
        summary = wandb.Table(columns=["policy", "success_rate", "ci_low", "ci_high", "avg_max_reward"])
        for label, r in runs.items():
            k = sum(r["successes"])
            lo, hi = wilson_ci(k, n)
            summary.add_data(label, k / n, lo, hi, sum(r["max_rewards"]) / n)
        per_seed = wandb.Table(columns=["policy", "seed", "success", "max_reward", "video"])
        for label, r in runs.items():
            for i, seed in enumerate(seeds):
                video = r["videos"].get(i)
                per_seed.add_data(label, seed, r["successes"][i], r["max_rewards"][i],
                                  wandb.Video(video, format="mp4") if video and Path(video).exists() else None)
        run.log({"summary": summary, "per_seed": per_seed})
        run.finish()


if __name__ == "__main__":
    main()

In [ ]:
runs = [f'--run "ACT ({STEPS // 1000}k steps, ours)={EVAL_DIR}/act"',
        f'--run "Diffusion Policy (lerobot/diffusion_pusht)={EVAL_DIR}/diffusion"']
if (EVAL_DIR / "act_ensemble" / "eval_info.json").exists():
    runs.append(f'--run "ACT + temporal ensembling={EVAL_DIR}/act_ensemble"')
RUN_ARGS = " ".join(runs)
!python /content/compare_results.py {RUN_ARGS} --start-seed {EVAL_START_SEED} --out {WORK}/results --wandb-project {WANDB_PROJECT}

## 8. Look at failure cases and save everything to Drive

In [ ]:
from IPython.display import Video, Markdown, display
display(Markdown((WORK / "results" / "failures.md").read_text()))

In [ ]:
# Show up to 3 failed episodes that have a video, per policy.
import json
for name in ["act", "diffusion", "act_ensemble"]:
    info_path = EVAL_DIR / name / "eval_info.json"
    if not info_path.exists():
        continue
    m = json.loads(info_path.read_text())["per_task"][0]["metrics"]
    shown = 0
    for i, path in enumerate(m.get("video_paths", [])):
        if not m["successes"][i] and shown < 3:
            print(f"{name}: seed {EVAL_START_SEED + i}, max reward {m['max_rewards'][i]:.2f}")
            display(Video(path, embed=True, width=320))
            shown += 1

In [ ]:
shutil.copytree(WORK / "results", DRIVE_DIR / "results", dirs_exist_ok=True)
shutil.copytree(EVAL_DIR, DRIVE_DIR / "eval", dirs_exist_ok=True)
print("Saved results and eval videos to", DRIVE_DIR)
print((WORK / "results" / "results.md").read_text())

## Done

Results are in `results/` (also copied to Drive): `results.md`, `per_seed.csv`, `failures.md`. Eval videos are under `eval/<policy>/videos/`.